In [71]:
import hmac

import numpy as np
import hashlib
import os

from cryptography.hazmat.primitives.asymmetric import x25519
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

# User Registration

Initialization of users Alice and Bob

the registration phase consists of a party generating a key bundle consisting of:

1. one identity key (long term)
2. one signed pre-key (mid term)
3. multiple ephemeral keys (short term)

Key signature is done using the identity key

In [72]:
class KeyBundle:
    def __init__(self):
        self.identity_key = x25519.X25519PrivateKey.generate()
        self.signed_pre_key = x25519.X25519PrivateKey.generate()
        self.ephemeral_keys = [x25519.X25519PrivateKey.generate() for _ in range(10)]


    def get_public_keys(self):
        return self.identity_key.public_key(), self.signed_pre_key.public_key(), self.ephemeral_keys

In [73]:
alice = KeyBundle()
bob = KeyBundle()

# Extended Triple Diffie Hellman

Alice computes her master secret

In [74]:
alice_dh_1 = alice.identity_key.exchange(bob.signed_pre_key.public_key())
alice_dh_2 = alice.ephemeral_keys[0].exchange(bob.identity_key.public_key())
alice_dh_3 = alice.ephemeral_keys[0].exchange(bob.signed_pre_key.public_key())
alice_dh_4 = alice.ephemeral_keys[0].exchange(bob.ephemeral_keys[0].public_key()) # will not be necessary if alice exhausts her ephemeral keys

alice_ms = alice_dh_1 + alice_dh_2 + alice_dh_3 + alice_dh_4


Bob computes his master secret

In [75]:
bob_dh_1 = bob.signed_pre_key.exchange(alice.identity_key.public_key())
bob_dh_2 = bob.identity_key.exchange(alice.ephemeral_keys[0].public_key())
bob_dh_3 = bob.signed_pre_key.exchange(alice.ephemeral_keys[0].public_key())
bob_dh_4 = bob.ephemeral_keys[0].exchange(alice.ephemeral_keys[0].public_key())

bob_ms = bob_dh_1 + bob_dh_2 + bob_dh_3 + bob_dh_4

Derivating the master secret:

In [76]:
def key_derivation_function_root(master_secret):
    return HKDF(
        algorithm=hashes.SHA256(),
        length=64,
        salt=b'\x00' * 32,
        info=b"X3DH_algr",
    ).derive(master_secret)

In [77]:
alice_keys =  key_derivation_function_root(master_secret=alice_ms)
alice_root_key, alice_chain_key = alice_keys[:32], alice_keys[32:]

In [78]:
bob_keys = key_derivation_function_root(master_secret=bob_ms)
bob_root_key, bob_chain_key = bob_keys[:32], bob_keys[32:]

In [79]:
print(alice_root_key.hex())
print(bob_root_key.hex())

e55734189cef9d16890662ff59bd9bc96ea4ed7aeed6ed1dff65b4d2f818498c
e55734189cef9d16890662ff59bd9bc96ea4ed7aeed6ed1dff65b4d2f818498c


# Symmetric Ratcheting

In [80]:
def kdf_message(chain_key):
    msg_key = hmac.new(chain_key, b"\x01", hashlib.sha256).digest()
    next_chain_key = hmac.new(chain_key, b"\x02", hashlib.sha256).digest()
    return msg_key, next_chain_key

In [81]:
alice_chain_key, msg_key_1 = kdf_message(alice_chain_key)
alice_chain_key, msg_key_2 = kdf_message(alice_chain_key)

print(msg_key_1.hex())
print(msg_key_2.hex())

0ed0cab24134db4f7098b14b1800859e6d1ee273d70a1be53d4724b42dcb9466
2c5d501dd93e295d2005ebf51b364d362bcbdc8d48d5678c377a7c90c1dda93b


# Asymmetric Ratcheting

On initialization:

`root_key, chaining_key = kdf_root(master_secret, bob.shared_secret.exchange(alice.ratchet_key.public_key()))`

In [82]:
def kdf_root(root_key, shared_secret):
    result = hmac.new(root_key, shared_secret, hashlib.sha256).digest()
    return result[:16], result[16:]

In [83]:
alice_ratchet = x25519.X25519PrivateKey.generate()
alice_root_key, alice_chain_key = kdf_root(alice_root_key, alice_ratchet.exchange(bob.identity_key.public_key()))
print(alice_root_key.hex())


e55734189cef9d16890662ff59bd9bc96ea4ed7aeed6ed1dff65b4d2f818498c
59d279c50ee46c63aa9d77596e511314


From state initiator-receiver to receiver-initiator

```
temp, chaining_key = kdf_root(root_key, alice.ratchet_key_prev.exchange(bob.ratchet_key.public_key())
```

In [84]:
bob_ratchet_keys = x25519.X25519PrivateKey.generate()

temp, alice_chain_key_ri = kdf_root(alice_root_key, alice_ratchet.exchange(bob_ratchet_keys.public_key()))

print(alice_chain_key_ri.hex())

d14327fa66573f8b569931c993dce2ae


From state receiver-initiator to initiator-receiver

```
root_key, chaining_key = kdf_root(temp, alice.ratchet_key.exchange(bob_ratchet_keys.public_key()))
```

In [85]:
alice_new_ratchet_keys = x25519.X25519PrivateKey.generate()

new_alice_root_key, alice_chain_key_ir = kdf_root(temp, alice_new_ratchet_keys.exchange(bob_ratchet_keys.public_key()))

print(new_alice_root_key.hex())

b4ce7cc63a633bac39e0c42b8bc85394
